In [ ]:
import re

def parse_sid(sid_text):
    print("inside parse_sid")
    pattern = re.compile(r"\d+\.\s*(.+?)\s*->\s*(.+?):\s*(.+)")
    sid = []
    for line in sid_text.splitlines():
        match = pattern.match(line.strip())
        if match:
            sender = match.group(1).strip()
            receiver = match.group(2).strip()
            message = match.group(3).strip()
            sid.append((f"{sender} -> {receiver}", message))
    return sid

In [ ]:
##############SID Graph##########################
import os


from graphviz import Digraph
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_md")  # Make sure this is installed

# Define canonical entities
canonical_entities = [
    "Customer",
    "E-commerce Website"    
]

# Convert to spaCy Docs for similarity comparison
canonical_docs = {ent: nlp(ent) for ent in canonical_entities}

def normalize_entity(entity):
    entity_doc = nlp(entity)
    best_match = None
    best_score = 0.75  # Similarity threshold (tune as needed)

    for canon_name, canon_doc in canonical_docs.items():
        score = entity_doc.similarity(canon_doc)
        if score > best_score:
            best_match = canon_name
            best_score = score

    return best_match if best_match else entity.strip()

def draw_sid_graph(sid_list, output_path="sid_graph"):
    dot = Digraph(comment="SID - Sequence Interaction Diagram")
    dot.attr(dpi='300')  # increase DPI for sharper text
    dot.attr(fontsize='14')  
    dot.attr(rankdir='LR',splines='polyline')  # Left to right direction
       
    # Deduplicate using a set
    normalized_edges = set()

    for interaction, action in sid_list:
        source, target = [normalize_entity(s.strip()) for s in interaction.split("->")]
        key = (source, target, action.strip())

        if key in normalized_edges:
            print(f"Skipping duplicate: {source} -> {target} [{action}]")
            continue

        normalized_edges.add(key)
        dot.node(source, source, shape="box", style="filled", fillcolor="lightblue")
        dot.node(target, target, shape="box", style="filled", fillcolor="lightgreen")
        dot.edge(source, target, label=action)

    # Render graph to file
    dot.render(output_path, format='png', cleanup=True)
    print(f"SID graph saved to: {output_path}.png")

In [ ]:
def extract_sbd_section(full_text):
    lines = full_text.splitlines()
    sbd_start = None
    for i, line in enumerate(lines):
         if "SBD" in line.upper():
            sbd_start = i + 1  # start after this line
            break
    if sbd_start is None:
        print("No SBD section found!")
        return ""

    # Extract until explanation or EOF
    sbd_lines = []
    for line in lines[sbd_start:]:
        if line.strip().startswith("### Explanation"):
            break
        sbd_lines.append(line)

    return "\n".join(sbd_lines).strip()

In [ ]:
from graphviz import Digraph
import re
import os

import re

def parse_sbd_(sbd_text):
    """
    Parses the SBD text into a dict of {subject_name: sbd_text_for_that_subject}
    Assumes subjects are marked by lines like: '#### Subject Name:'
    Returns: dict of subject -> string (SBD text)
    """
    subjects = {}
    current_subject = None
    current_lines = []

    for line in sbd_text.splitlines():
       
        subject_match = re.match(r'^\s*####\s*(.+?):\s*$', line)
        if subject_match:
            # Save the previous subject's lines
            if current_subject is not None and current_lines:
                subjects[current_subject] = "\n".join(current_lines).strip()

            # Start a new subject
            current_subject = subject_match.group(1).strip()
            current_lines = []
        else:
            if current_subject is not None:
                current_lines.append(line)

    # Save the last subject block
    if current_subject is not None and current_lines:
        subjects[current_subject] = "\n".join(current_lines).strip()

    return subjects

In [ ]:
from graphviz import Digraph
import re

def draw_sbd_graph(sbd_text, output_path):
    """
    Parse SBD text and create a Graphviz SBD diagram.
    Saves as PDF at `output_path`.
    """
    dot = Digraph("SBD", format='pdf')  
    dot.attr(rankdir='LR')
    
    def add_edge_safe(src, dst, label=""):
        if dst is not None and dst != src:
            dot.edge(str(src), str(dst), label=label)

    state_styles = {
        'StartState': {"shape": "rectangle", "color": "yellow"},
        'EndState':   {"shape": "rectangle", "color": "yellow"},
        'SendState':  {"shape": "rectangle", "color": "green"},
        'ReceiveState': {"shape": "rectangle", "color": "pink"},
        'DoState':    {"shape": "rectangle", "color": "yellow"},
        'Unknown':    {"shape": "rectangle", "color": "lightgrey"},
    }

    steps = {}
    lines = sbd_text.strip().splitlines()
    current_step = None
    i = 0

    while i < len(lines):
        line = lines[i].strip()
        match = re.match(r'^(\d+)\.\s*\*{0,2}(\w+)\*{0,2}:\s*(.*)', line)
        if match:
            num, state_type, label = match.groups()
            current_step = int(num)

            if state_type == "GotoStep":
                target_step = int(label.strip().split()[0])
                steps[current_step] = {
                    "type": "GotoStep",
                    "label": "",
                    "choices": [],
                    "branches": [],
                    "goto": target_step
                }
            else:
                steps[current_step] = {
                    "type": state_type,
                    "label": label,
                    "choices": [],
                    "branches": [],
                    "goto": None
                }

                # Lookahead for extra details
                if state_type == "SendState":
                    if i + 1 < len(lines) and lines[i + 1].strip().startswith("To:"):
                        steps[current_step]["To"] = lines[i + 1].strip().split(":", 1)[1].strip()
                        i += 1
                    if i + 1 < len(lines) and lines[i + 1].strip().startswith("Msg:"):
                        steps[current_step]["Msg"] = lines[i + 1].strip().split(":", 1)[1].strip()
                        i += 1
                    if i + 1 < len(lines) and lines[i + 1].strip().startswith("Next:"):
                        next_val = lines[i + 1].strip().split(":", 1)[1].strip()
                        if "End of process" in next_val:
                            steps[current_step]["next"] = "END"
                        else:
                            steps[current_step]["next"] = int(next_val)
                        i += 1

                if state_type in {"StartState", "DoState"}:
                    if i + 1 < len(lines) and lines[i + 1].strip().startswith("OutgoingLabel:"):
                        steps[current_step]["OutgoingLabel"] = lines[i + 1].strip().split(":", 1)[1].strip()
                        i += 1
                    elif i + 1 < len(lines) and lines[i + 1].strip().startswith("Description:"):
                        steps[current_step]["OutgoingLabel"] = lines[i + 1].strip().split(":", 1)[1].strip()
                        i += 1
               
                # --- HANDLE ReceiveState (multi-line lookahead) ---
                if state_type == "ReceiveState":
                    # scan subsequent lines until hit a new numbered step or an empty line
                    j = i + 1
                    while j < len(lines):
                        nxt = lines[j].strip()
                        # stop if next line is the start of a new numbered state
                        if re.match(r'^\d+\.\s*\*{0,2}\w+\*{0,2}:\s*', nxt):
                            break

                        # '- From:' or 'From:' (support both bullet and no-bullet formats)
                        if re.match(r'^-?\s*From:', nxt, re.IGNORECASE):
                            from_actor = nxt.split(':', 1)[1].strip()
                            msg = ""
                            next_step = None
                            if j + 1 < len(lines) and re.match(r'^\s*Msg:', lines[j + 1], re.IGNORECASE):
                                msg = lines[j + 1].split(':', 1)[1].strip()
                                j += 1
                            if j + 1 < len(lines) and re.match(r'^\s*Next:', lines[j + 1], re.IGNORECASE):
                                next_step = int(lines[j + 1].split(':', 1)[1].strip())
                                j += 1
                            steps[current_step]["choices"].append({
                                "from": from_actor,
                                "msg": msg,
                                "next": next_step
                            })
                        # case: 'Msg:' then 'Next:' (no From)
                        elif nxt.startswith('Msg:'):
                            msg = nxt.split(':', 1)[1].strip()
                            next_step = None
                            if j + 1 < len(lines) and lines[j + 1].strip().startswith('Next:'):
                                next_step = int(lines[j + 1].strip().split(':', 1)[1].strip())
                                j += 1
                            steps[current_step]["choices"].append({
                                "from": "",
                                "msg": msg,
                                "next": next_step
                            })
                        # case: direct 'Next:' (no From/Msg)
                        elif nxt.startswith('Next:'):
                            next_val = nxt.split(':', 1)[1].strip()
                            if "End of process" in next_val:
                                next_step = "END"
                            else:
                                next_step = int(next_val)
                            steps[current_step]["choices"].append({
                                "from": "",
                                "msg": "",
                                "next": next_step
                            })
                        # other lines: skip (or break on blank line)
                        elif nxt == "" :
                            break
                        j += 1

                    # advance i so the main loop continues after the consumed lookahead lines
                    i = j - 1

            i += 1
            continue
            
           

        
          
        
        
        # Parse DoState branches
       
        elif line.startswith('- Step:') and current_step is not None:
            step_str = line.split(':', 1)[1].strip()
            try:
                step = int(step_str)
            except ValueError:
                # Handle non-integer step (like descriptive text)
                step = None

            desc = ""
            if i + 1 < len(lines) and lines[i + 1].strip().startswith('Description:'):
                desc = lines[i + 1].strip().split(':', 1)[1].strip()
                i += 1

            steps[current_step]["branches"].append({
                "step": step,
                "desc": desc or step_str  # use text as description if step is None
            })
        elif 'GotoStep' in line and current_step is not None:
            match = re.match(r'\*{0,2}GotoStep\*{0,2}:\s*(\d+)', line)
            if match:
                steps[current_step]["goto"] = int(match.group(1))

        i += 1

    # Add missing goto targets
    goto_targets = {info["goto"] for info in steps.values() if info["goto"] is not None}
    for tgt in goto_targets:
        if tgt not in steps:
            steps[tgt] = {
                "type": "Unknown",
                "label": f"Step {tgt} \\nundefined",
                "choices": [],
                "branches": [],
                "goto": None
            }

    
    # Find all reachable steps
    reachable = set()
    for num, info in steps.items():
        # Direct next
        if "next" in info and info["next"] is not None:
            reachable.add(info["next"])
        # ReceiveState choices
        for c in info.get("choices", []):
            if c.get("next") is not None:
                reachable.add(c["next"])
        # DoState branches
        for b in info.get("branches", []):
            reachable.add(b["step"])
        # GotoStep
        if info.get("goto") is not None:
            reachable.add(info["goto"])
        
    #include sequential fallback targets
    all_step_nums = sorted(steps.keys())
    for idx, num in enumerate(all_step_nums):
        info = steps[num]
        next_step = all_step_nums[idx + 1] if idx + 1 < len(all_step_nums) else None
        uses_fallback = False
        if info["type"] == "StartState":
            uses_fallback = True
        elif info["type"] == "SendState" and info.get("next") is None:
            uses_fallback = True
        elif info["type"] == "DoState" and not info.get("branches"):
            uses_fallback = True
        if uses_fallback and next_step is not None:
            reachable.add(next_step)
            
    # If any step references the special "END", add an End node
    has_end = any(
        (("next" in info and info["next"] == "END") or
         any(c.get("next") == "END" for c in info.get("choices", [])))
        for info in steps.values()
    )
    if has_end:
        dot.node("END", label="End", shape="rectangle", style="filled", fillcolor="yellow")       
  
    # Draw only reachable + start nodes
    for num, info in steps.items():
        if num != min(steps.keys()) and num not in reachable:
            continue  # skip orphans except the start step

        if info["type"] == "GotoStep":
            continue
        if info["type"] == "Unknown" and info["label"].endswith("undefined"):
            continue
            
      

        style = state_styles.get(info["type"], {"shape": "rectangle", "color": "lightgrey"})
        fill = style["color"]
       
       
        
        #StartState color rule: based on its own label 
        if info["type"] == "StartState":
            lbl = (info.get("label") or "").lower()
            # simple substring check handles 'receive', 'receiving', 'send', 'sending'
            if "receive" in lbl:
                fill = state_styles["ReceiveState"]["color"]   # pink
            elif "send" in lbl:
                fill = state_styles["SendState"]["color"]      # green
            else:
                fill = "yellow"                                # default for do/other
        if info["type"] == "DoState":
               fill = "yellow"

        dot.node(str(num),
                 label=info["label"],
                 shape=style["shape"],
                 style="filled",
                 fillcolor=fill)

    # Draw edges
    all_step_nums = sorted(steps.keys())
    for idx, num in enumerate(all_step_nums):
        info = steps[num]

        if info["type"] == "GotoStep":
            continue
        if info["type"] == "EndState":
            continue

        
        if info["type"] == "ReceiveState":
            if info["choices"]:
                # Only draw from choices
                for choice in info["choices"]:
                    label = f"From: {choice['from']}\\nMsg: {choice['msg']}"
                    if choice["next"] is not None:
                        
                        add_edge_safe(num, choice["next"], label)
            else:
                # No choices, fall back to single Next
                if "next" in info and info["next"] is not None:
                    
                    add_edge_safe(num, info["next"], "")
           

        elif info["type"] == "SendState":
            label_parts = []
            if "To" in info:
                label_parts.append(f"To:{info['To']}")
            if "Msg" in info:
                label_parts.append(f"Msg:{info['Msg']}")
            label = "\\n".join(label_parts)

            # Prefer explicit Next: if present
            if "next" in info and info["next"] is not None:
                
                add_edge_safe(num, info["next"], label)
            
            else:
                # fallback to sequential
                next_step = all_step_nums[idx + 1] if idx + 1 < len(all_step_nums) else None
                if next_step:
                    
                    add_edge_safe(num, next_step, label)

        elif info["type"] == "StartState":
            next_step = all_step_nums[idx + 1] if idx + 1 < len(all_step_nums) else None
            if next_step:
                label = info.get("OutgoingLabel", info["label"])
               
                add_edge_safe(num, next_step, label)

        elif info["type"] == "DoState":
            if info["branches"]:
                for branch in info["branches"]:
                    target_step = branch["step"]
                    target_info = steps.get(target_step)
                    if target_info and target_info["type"] == "GotoStep" and target_info.get("goto"):
                        real_target = target_info["goto"]
                        
                        add_edge_safe(num, real_target, branch["desc"])
                    else:
                        
                        add_edge_safe(num, target_step, branch["desc"])
            else:
                next_step = all_step_nums[idx + 1] if idx + 1 < len(all_step_nums) else None
                if next_step:
                    label = info.get("OutgoingLabel", info["label"])
                    
                    add_edge_safe(num, next_step, label)

        if info["goto"]:
           
            add_edge_safe(num, info["goto"])
            continue
        
        else:
            next_step = all_step_nums[idx + 1] if idx + 1 < len(all_step_nums) else None
            has_outgoing = (
                bool(info["choices"]) or
                bool(info["branches"]) or
                bool(info["goto"]) or
                info["type"] in {"SendState", "DoState", "StartState"}
            )

            if next_step and not has_outgoing and info["type"] not in {"ReceiveState"}:
                
                add_edge_safe(num, next_step, "(default)")

    # Save final graph
    dot.render(output_path, cleanup=True)
    print(f"SBD graph saved at {output_path}.pdf")

In [ ]:
def draw_all_subjects(sbd_text, output_folder):
    """
    Parses multi-subject SBD text, draws a graph per subject, and saves PDFs
    in output_folder with filenames as subject names.
    """
    print("Inside draw all subjects")
    subjects = parse_sbd_(sbd_text)
    print("Subjects found:", list(subjects.keys()))
    print("subjects: ",subjects)

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for subject, subject_sbd_text in subjects.items():
        safe_subject = re.sub(r'[^\w\-_. ]', '_', subject).replace(' ', '_')
        output_path = os.path.join(output_folder, safe_subject) 
       
        draw_sbd_graph(subject_sbd_text, output_path)

In [ ]:
from rdflib import Graph, Namespace, Literal 
from rdflib.namespace import RDF, RDFS, OWL, XSD
import re
from textwrap import dedent
from typing import Optional


# Namespaces
ABSTRACT = Namespace("http://www.imi.kit.edu/abstract-pass-ont#")
STANDARD = Namespace("http://www.i2pm.net/standard-pass-ont#")
BASE     = Namespace("http://subjective-me.jimdo.com/s-bpm/processmodels/2025-03-25/Page-1#")

def sid_to_pass_owl(llama_text: str,
                    model_label: str = "PASS_Model",
                    out_file: Optional[str] = None) -> str:
    print("Inside OWL Function")
    # STEP 1: Parse Subjects and SID lines from LLM text ---
    subjects = []
    subj_mode = False
    sid_lines = []
    sid_mode = False

    for ln in llama_text.splitlines():
        ln_strip = ln.strip()

        if ln_strip.startswith("### Subjects"):
            subj_mode = True
            sid_mode = False
            continue

        # Stop subject parsing when hitting next section
        if ln_strip.startswith("### SID") or ln_strip.startswith("### Subject Interaction Diagram"):
            subj_mode = False
            sid_mode = True
            continue

        if subj_mode and ln_strip.startswith("-"):
            subjects.append(ln_strip[1:].strip())

        if sid_mode and re.match(r"\d+\.", ln_strip):
            sid_lines.append(ln_strip)
        
      
    # STEP 2: Build RDF graph
    g = Graph(base=BASE)
    g.bind("abstract-pass-ont", ABSTRACT)
    g.bind("standard-pass-ont", STANDARD)
    g.bind("owl", OWL)
    g.bind("rdfs", RDFS)
    g.bind("xsd", XSD)

    # Add PASSProcessModel individual
    model_uri = BASE[model_label]
    g.add((model_uri, RDF.type, STANDARD.PASSProcessModel))
    g.add((model_uri, STANDARD.hasModelComponentID, Literal(f"{model_uri}#Model", datatype=XSD.string)))
    g.add((model_uri, STANDARD.hasModelComponentLabel, Literal(model_label, lang="en")))

    sid_layer = BASE["SID_1"]
    g.add((sid_layer, RDF.type, ABSTRACT.ModelLayer))
    g.add((sid_layer, STANDARD.hasModelComponentID, Literal("SID_1", datatype=XSD.string)))
    g.add((sid_layer, STANDARD.hasModelComponentLabel, Literal("SID_1", lang="en")))
    g.add((sid_layer, STANDARD.hasPriorityNumber, Literal(1, datatype=XSD.positiveInteger)))
    g.add((model_uri, STANDARD.contains, sid_layer))

    # STEP 3: Subjects
    subj_id_map = {}
    print("subjects",subjects)
    for idx, subj_label in enumerate(subjects, start=2):
        sid = f"SID_1_FullySpecifiedSubject_{idx}"
        subj_uri = BASE[sid]
        subj_id_map[subj_label.strip()] = subj_uri  
        

        g.add((subj_uri, RDF.type, STANDARD.FullySpecifiedSubject))
        g.add((subj_uri, STANDARD.hasModelComponentID, Literal(sid, datatype=XSD.string)))
        g.add((subj_uri, STANDARD.hasModelComponentLabel, Literal(subj_label, lang="en")))
        g.add((subj_uri, STANDARD.hasMaximumSubjectInstanceRestriction, Literal(1, datatype=XSD.integer)))
        g.add((subj_uri, ABSTRACT.hasExecutionCostPerHour, Literal(0.0, datatype=XSD.double)))

        g.add((sid_layer, STANDARD.contains, subj_uri))
        g.add((model_uri, STANDARD.contains, subj_uri))

    # STEP 4: Process SID messages
    mel_counter = 1
    msg_counter = 1
    for line in sid_lines:
        print("line",line)
        m = re.match(r"\d+\.\s*(.+?)\s*->\s*(.+?):\s*(.+)", line)
        if not m:
            continue
        sender, receiver, msg = m.groups()
        sender_uri   = subj_id_map[sender.strip()]
        receiver_uri = subj_id_map[receiver.strip()]

        msg_spec_id = f"SID_1_MessageSpecification_{msg_counter}"
        msg_spec_uri = BASE[msg_spec_id]
        g.add((msg_spec_uri, RDF.type, STANDARD.MessageSpecification))
        g.add((msg_spec_uri, STANDARD.hasModelComponentID, Literal(msg_spec_id, datatype=XSD.string)))
        g.add((msg_spec_uri, STANDARD.hasModelComponentLabel, Literal(msg, lang="en")))

        payload_id = f"PayloadDefinition_of_{msg_spec_id}"
        payload_uri = BASE[payload_id]
        g.add((payload_uri, RDF.type, OWL.Class))
        g.add((msg_spec_uri, STANDARD.containsPayloadDescription, payload_uri))

        mel_id = f"MessageExchangeList_on_SID_1_StandardMessageConnector_{mel_counter}"
        mel_uri = BASE[mel_id]
        conn_id = f"SID_1_StandardMessageConnector_{mel_counter}"
        conn_uri = BASE[conn_id]

        g.add((mel_uri, RDF.type, STANDARD.MessageExchangeList))
        g.add((mel_uri, STANDARD.hasModelComponentID, Literal(mel_id, datatype=XSD.string)))
        g.add((mel_uri, STANDARD.hasModelComponentLabel, Literal(conn_id, lang="en")))
        g.add((mel_uri, STANDARD.contains, msg_spec_uri))

        g.add((conn_uri, RDF.type, STANDARD.StandardMessageConnector))
        g.add((conn_uri, STANDARD.hasSender, sender_uri))
        g.add((conn_uri, STANDARD.hasReceiver, receiver_uri))
        g.add((conn_uri, STANDARD.hasMessageType, msg_spec_uri))
        g.add((mel_uri, STANDARD.contains, conn_uri))

        for parent in (sid_layer, model_uri):
            g.add((parent, STANDARD.contains, mel_uri))
            g.add((parent, STANDARD.contains, msg_spec_uri))
            g.add((parent, STANDARD.contains, conn_uri))

        mel_counter += 1
        msg_counter += 1  
    
    # === Detect ### SBD section first ===
    # STEP 5: Parse SBD section
    sbd_text_lines = []
    in_sbd_section = False
    for ln in llama_text.splitlines():
        ln_strip = ln.strip()
        if ln_strip.startswith("### SBD") or ln_strip.startswith("### Subject Behavior Diagram"):
            in_sbd_section = True
            continue
        
        if in_sbd_section:
            sbd_text_lines.append(ln)

    sbd_sections = {}
    current_subject = None
    current_lines = []

    for ln in sbd_text_lines:
        ln_strip = ln.strip()
        if ln_strip.startswith("#### "):
            # Save the previous block
            if current_subject and current_lines:
                # Always overwrite with the latest version
                sbd_sections[current_subject] = current_lines[:]
                print(f"Overwriting SBD for subject-1: {current_subject}")

         
            
            if m := re.match(r"^####\s*(.+?):", ln_strip):
                subj = m.group(1).strip()
                subj = subj.split("(")[0].strip()  # normalize parentheses
                current_subject = subj
                current_lines = []

           
        elif current_subject:
            current_lines.append(ln)

    # Save the last block
    if current_subject and current_lines:
        print(f"Overwriting SBD for subject-2: {current_subject}")
        sbd_sections[current_subject] = current_lines[:]
        
    # --- CLEAN empty lines in each SBD section ---
    for subj, lines in sbd_sections.items(): 
        sbd_sections[subj] = [ln for ln in lines if ln.strip()]


    # STEP 6: LOOP through EACH SBD section
    sbd_index = 1
    print("sbd_sections",sbd_sections)
    for sbd_name, sbd_lines in sbd_sections.items():
        subj_uri = subj_id_map.get(sbd_name.strip())
        if not subj_uri:
            print(f"Warning: No matching FullySpecifiedSubject for {sbd_name}, skipping SBD")
            continue
        steps = {}
        current_step = None
        subj_sid_idx = subj_uri.split("_")[-1]
        sbd_id = f"SBD_{sbd_index}_SID_1_FullySpecifiedSubject_{subj_sid_idx}"
        sbd_uri = BASE[sbd_id]
        g.add((sbd_uri, RDF.type, STANDARD.SubjectBehavior))
        g.add((sbd_uri, STANDARD.hasModelComponentID, Literal(sbd_id, datatype=XSD.string)))
        g.add((sbd_uri, STANDARD.hasModelComponentLabel, Literal(f"SBD: {sbd_name}", lang="en")))
        g.add((sbd_uri, STANDARD.hasPriorityNumber, Literal(sbd_index, datatype=XSD.positiveInteger)))
        g.add((subj_uri, STANDARD.hasBehavior, sbd_uri))

        # --- Parse numbered states ---
        sbd_blocks = []
        current_block = []
        for ln in sbd_lines:
            if re.match(r"\d+\.", ln.strip()):
                if current_block:
                    sbd_blocks.append(current_block)
                current_block = [ln.strip()]
            else:
                if current_block:
                    current_block.append(ln.strip())
        if current_block:
            sbd_blocks.append(current_block)

        # --- Build lookup ---
        state_lookup = {}
        connections = []

        for block in sbd_blocks:
            m = re.match(r"(\d+)\.\s*(\w+State|GotoStep):\s*(.*)", block[0])
            if not m:
                continue
            step_num, state_type, label = m.groups()
            step_num = int(step_num)
            current_step = step_num
            steps[current_step] = {"choices": []}

            desc_val, to_val, from_val, msg_val = None, None, None, None
            next_steps = []
            branches = []
            choices = []
            goto_target = None

            in_choices = False
            in_branches = False

            for line in block[1:]:
                if m_desc := re.match(r"Description:\s*(.*)", line):
                    desc_val = m_desc.group(1).strip()
                if m_to := re.match(r"To:\s*(.*)", line):
                    to_val = m_to.group(1).strip()
                if m_from := re.match(r"From:\s*(.*)", line):
                    from_val = m_from.group(1).strip()
                if m_msg := re.match(r"Msg:\s*(.*)", line):
                    msg_val = m_msg.group(1).strip()
                if m_next := re.match(r"Next:\s*(\d+)", line):
                    try:
                        next_steps.append(int(m_next.group(1)))
                    except ValueError:
                        # Skip non-numeric Next references
                        continue
                if m_goto := re.match(r"GotoStep:\s*(\d+)", line):
                    goto_target = int(m_goto.group(1))

                # Choices start
                if line.startswith("Choices:"):
                    in_choices = True
                    continue
                if in_choices and line.startswith("Branches:"):
                    in_choices = False
                # parsing choices
                if in_choices and line.strip().startswith("-"):
                    # get current line index in block
                    idx = block[1:].index(line)
                    subblock = block[1:]  # lines being iterated over
                    # old-style From/Msg/Next
                    if m_from2 := re.match(r"-\s*From:\s*(.*)", line):
                        choice_from = m_from2.group(1).strip()
                        choice_msg = ""
                        choice_next = None
                        if idx + 1 < len(subblock) and subblock[idx + 1].strip().startswith("Msg:"):
                            choice_msg = subblock[idx + 1].split(":",1)[1].strip()
                            
                        if idx + 1 < len(subblock) and subblock[idx + 1].strip().startswith("Next:"):
                            choice_next = int(subblock[idx + 1].split(":",1)[1].strip())
                            
                    # New: Handle label: Next: number format
                    elif m_label_next := re.match(r"-\s*(.*):\s*Next:\s*(\d+)", line):
                        choice_from = ""
                        choice_msg = m_label_next.group(1).strip()  # text before Next
                        choice_next = int(m_label_next.group(2))
                    else:
                        continue  # ignore unknown format

                    steps[current_step]["choices"].append({
                        "from": choice_from,
                        "msg": choice_msg,
                        "next": choice_next
                    })
                

                # Branches start
                if line.startswith("Branches:"):
                    in_branches = True
                    continue
                if in_branches and line.strip().startswith("-"):
                    if m_step := re.match(r"-\s*Step:\s*(\d+)", line):
                        branches.append(int(m_step.group(1)))
                    else:
                        # Non-integer branch, store as annotation
                        desc_annotation = line.split(":", 1)[1].strip()
                       
            state_lookup[step_num] = {
                "type": state_type,
                "label": label,
                "desc": desc_val,
                "to": to_val,
                "from": from_val,
                "msg": msg_val
            }

            for n in next_steps:
                connections.append((step_num, n, None, None))
            for b in branches:
                connections.append((step_num, b, None, None))
            for cf, cm, cn in choices:
                connections.append((step_num, cn, cf, cm))
            if goto_target:
                connections.append((step_num, goto_target, None, None))
                

            # --- Create the state individual ---
            state_id = f"SBD_{sbd_index}_{state_type}_{step_num}"
            state_uri = BASE[state_id]
            g.add((state_uri, RDF.type, STANDARD[state_type]))
            g.add((state_uri, STANDARD.hasModelComponentID, Literal(state_id, datatype=XSD.string)))
            g.add((state_uri, STANDARD.hasModelComponentLabel, Literal(label, lang="en")))
            g.add((sbd_uri, STANDARD.contains, state_uri))

        # --- Create transitions ---
        transition_counter = 1
        for source_step, target_step, choice_from, choice_msg in connections:
            src_info = state_lookup[source_step]
            tgt_info = state_lookup[target_step]

            trans_id = f"SBD_{sbd_index}_{src_info['type']}Transition_{transition_counter}"
            trans_uri = BASE[trans_id]

            if src_info['type'] == "SendState":
                trans_type = STANDARD.SendTransition
                trans_label = f"To: {src_info['to']}\nMsg: {src_info['msg']}"
            elif src_info['type'] == "ReceiveState":
                trans_type = STANDARD.ReceiveTransition
                if choice_from or choice_msg:
                    # Choice-specific transition
                    trans_label = f"(Choice) From: {choice_from or src_info['from']}\nMsg: {choice_msg or src_info['msg']}"
                else:
                    # Normal transition
                    trans_label = f"From: {src_info['from']}\nMsg: {src_info['msg']}"
            elif src_info['type'] == "GotoStep":
                trans_type = STANDARD.DoTransition
                trans_label = f"Go to step {target_step}"
            else:
                trans_type = STANDARD.DoTransition
                trans_label = src_info['desc'] or "Continue Process"

            g.add((trans_uri, RDF.type, trans_type))
            g.add((trans_uri, STANDARD.hasModelComponentID, Literal(trans_id, datatype=XSD.string)))
            g.add((trans_uri, STANDARD.hasModelComponentLabel, Literal(trans_label, lang="en")))
            g.add((trans_uri, STANDARD.hasSourceState, BASE[f"SBD_{sbd_index}_{src_info['type']}_{source_step}"]))
            g.add((trans_uri, STANDARD.hasTargetState, BASE[f"SBD_{sbd_index}_{tgt_info['type']}_{target_step}"]))
            g.add((sbd_uri, STANDARD.contains, trans_uri))
            transition_counter += 1

        g.add((sid_layer, STANDARD.contains, sbd_uri))
        g.add((model_uri, STANDARD.contains, sbd_uri))
        sbd_index += 1
        
    # example class
    dm_class = BASE["VisioShapesInternalDataMappingFunction"]
    g.add((dm_class, RDF.type, OWL.Class))
    g.add((dm_class, RDFS.subClassOf, STANDARD.DataMappingFunction))

    # STEP 7: Serialize RDF/XML
    xml_body = g.serialize(format="application/rdf+xml")

    entities = dedent("""\
        <!DOCTYPE rdf:RDF [
            <!ENTITY owl "http://www.w3.org/2002/07/owl#" >
            <!ENTITY xsd "http://www.w3.org/2001/XMLSchema#" >
            <!ENTITY rdfs "http://www.w3.org/2000/01/rdf-schema#" >
            <!ENTITY abstract-pass-ont "http://www.imi.kit.edu/abstract-pass-ont#" >
            <!ENTITY standard-pass-ont "http://www.i2pm.net/standard-pass-ont#" >
            <!ENTITY rdf "http://www.w3.org/1999/02/22-rdf-syntax-ns#" >
        ]>
    """)
    xml_body_nohead = "\n".join(xml_body.splitlines()[1:])
    final_xml = f'<?xml version="1.0"?>\n{entities}\n{xml_body_nohead}'

    if out_file:
        with open(out_file, "w", encoding="utf-8") as f:
            f.write(final_xml)

    return final_xml

In [ ]:
import panel as pn
from openai import OpenAI
import re
import panel as pn
from PIL import Image
from io import BytesIO

# Add a Panel pane to display the image
image_view = pn.pane.PNG(width=700)

pn.extension()

# --- API setup ---
api_key = ""
base_url = "https://gpt.uni-muenster.de/v1"
model = "gpt-oss-120b"  # model used
client = OpenAI(api_key=api_key, base_url=base_url)

# --- Prompt Templates ---
extract_subjects_system_prompt = """
You are an expert in Subject-Oriented Business Process Modeling using the Parallel Activity Specification Schema (PASS).

You will be given a scenario from the user.

Only identify all the subjects involved.
Do NOT add any additional information about the subject in brackets.
Return as a simple bullet list:

### Subjects:
- Subject 1
- Subject 2
- ...
"""

extract_subjects_user_prompt = """ \"\"\"{scenario}\"\"\" """

# --- One-shot example prompt ---
sid_sbd_sytem_prompt = """
You are an expert in Subject-Oriented Business Process Modeling using PASS.

Your task is to generate a PASS model in two parts:
1. Subject Interaction Diagram (SID)
2. Subject Behavior Diagram (SBD)

You will be given:
- a scenario 
- a fixed list of subjects

General Constraints:
- Use ONLY the given subjects.
- Do NOT introduce new subjects. 
- All interactions must occur only between these subjects 


## 1. Subject Interaction Diagram (SID):
- Describe interactions between subjects using messages in noun form only
- Do NOT use verbs
- Each interaction must be between two of the given subjects
- Use noun phrases that represent the content of the interaction, not the action.
- Return a numbered list of interactions using the following format only:
  '### Subject Interaction Diagram (SID)
  1. Subject A -> Subject B: Message'
- SID messages must represent passive informational artifacts
- Do NOT model actions, assignments, decisions, or process steps as messages.

## 2. Subject Behavior Diagram (SBD):
- Start this section with:
  '### Subject Behavior Diagram (SBD)'

### General Rules
- Create ONE SBD per subject
- Use this exact heading format:
  '#### SubjectName:'

- Each SBD starts with step number 1
- Step numbers:
  - must be written using the format 'X.'
  - must be unique
  - must increase sequentially
  - must be referenced by a valid 'Next:'/'Branch':

- Every step that is defined MUST be reachable from at least one previous step via 'Next:' or 'Branches:'
- Do NOT create implicit transitions or placeholder steps (e.g. "skip")
- Every step must explicitly define a StateType.

- Not all interactions described in the scenario must occur in every execution path
- Interactions may be optional and must be modeled using branching behavior

- Every interaction in the SID must appear exactly ONCE as:
  - one 'SendState'
  - one 'ReceiveState'
- Message names MUST match the SID message name EXACTLY
- Subject names in SBD headers MUST match the given subject list EXACTLY

### State Types
Use exactly ONE of the following per step:
  - StartState
  - DoState
  - SendState
  - ReceiveState
  - EndState

EVERY state MUST include a descriptive title directly after the state type, using the format:
  'StateType: State Description'

The separate line 'Description:' is only allowed in StartState and DoState.

'Next:' may reference ANY step number, including earlier steps, to model loops or retries.

#### StartState
- Must include:
  'Description:'
- Describes what triggers the process or decision
- MUST be used once per SBD

#### SendState
- Must include:
  'To: Subject
   Msg: Message
   Next: X'
- Use SendStates only for explicit communication between subjects

#### ReceiveState
- Must include:
  'From: Subject
   Msg: Message
   Next: X'
- If multiple outcomes are possible, use:
  'Choices:
    - From:
      Msg:
      Next:'
- Each message may be received ONLY ONCE
- Do not mix unrelated messages in the same ReceiveState
- If the same message is needed after different branches, merge the control flow to that single ReceiveState

#### DoState
- Model internal activities, evaluations, diagnoses, and decisions exclusively as DoStates.
- If the DoState can have different outcomes:
  - Do NOT include a 'Description:' at the DoState level
  - Include:
    'Branches:
      - Step: X
        Description:
      - Step: X
        Description:'
- If the DoState represents internal activities, include:
  'Description:'

#### EndState
- Marks the end of a behavior path
- Every branch must end in its own EndState
- No SendState and ReceiveState may be the final step
- The State Description MUST be non-empty and describe the outcome of that behavior path

### Control Flow
- Use 'Next:' for all transitions, including loops and retries
- Loops and retries are modeled by referencing an earlier step number in 'Next:'
- Do NOT duplicate SendStates for retries, reuse them by pointing 'Next:' to the original SendState
- Do NOT introduce additional decision layers or interactions beyond those explicitly stated in the scenario

### Output Requirements
- Do NOT add explanations or commentary
- Use the same language as the scenario description and subject names for all textual elements
- Output ONLY:
  - the SID
  - the complete SBDs for all subjects
- Formatting is strict:
  - The "StateType: Title" line must contain ONLY the state type and title
  - "To:", "From:", "Msg:", "Next:", "Description:", "Branches:", "Choices:" MUST each be on their own line
  - Do NOT put multiple fields on one line
"""

sid_sbd_user_prompt = """
Scenario:
\"\"\"{scenario}\"\"\"

Finalized subjects:
{subjects}
"""

# --- Panel Widgets ---
scenario_input = pn.widgets.TextAreaInput(name='PASS Scenario', height=150, width=700)
extract_btn = pn.widgets.Button(name="Extract Subjects", button_type='primary')
generate_btn = pn.widgets.Button(name="Generate SID & SBD", button_type='success', disabled=True)

subjects_box = pn.widgets.CheckBoxGroup(name="Subjects", options=[], inline=False)
add_subject_input = pn.widgets.TextInput(name="Add New Subject", placeholder="e.g. Inventory System")
add_subject_btn = pn.widgets.Button(name="Add Subject", button_type='primary')
rename_input = pn.widgets.TextInput(name="Rename/Merge Subjects (comma separated)", placeholder="e.g. Website, App -> Customer Portal")
apply_merge_btn = pn.widgets.Button(name="Apply Merge/Rename", button_type='warning')

output_markdown = pn.pane.Markdown("### Output will appear here...", sizing_mode="stretch_width")


# --- Callbacks ---

def extract_subjects(event):
    print("Extract Subjects clicked!")
    scenario = scenario_input.value.strip()
    print(scenario)
    if not scenario:
        output_markdown.object = "Please enter a scenario description."
        return
    
    user_prompt = extract_subjects_user_prompt.replace("{scenario}", scenario)
    print(user_prompt)
    try:
        response = client.chat.completions.create(
          messages=[
            {
              "role": "system",
              "content": extract_subjects_system_prompt   
            },
            {
              "role": "user",
              "content": user_prompt
            }
          ],
          model=model,
          temperature=0.3
        )
        print(response.choices[0].message.content)
        raw_subjects = response.choices[0].message.content.strip()
        print("Raw subjects from model:", raw_subjects)
        # Parse subjects         
        subjects = []
        for line in raw_subjects.splitlines():
            line = line.strip()
            if line.startswith("-") or line.startswith("*"):
                subjects.append(line[1:].strip())
        
        subjects_box.options = subjects
        subjects_box.value = subjects  # pre-select all
        generate_btn.disabled = False
        output_markdown.object = f"### Extracted Subjects:\n{raw_subjects}"
    except Exception as e:
        output_markdown.object = f"API Error: {e}"

def apply_merge(event):
    merge_text = rename_input.value.strip()
    if "->" not in merge_text:
        return
    left, right = merge_text.split("->")
    old = [s.strip() for s in left.split(",")]
    new = right.strip()
    updated = [new if s in old else s for s in subjects_box.options]
    subjects_box.options = list(dict.fromkeys(updated))  # remove duplicates
    subjects_box.value = subjects_box.options
    
def add_subject(event):
    new_subject = add_subject_input.value.strip()
    if not new_subject:
        return
    # Avoid duplicates
    if new_subject not in subjects_box.options:
        subjects_box.options = subjects_box.options + [new_subject]
       
        subjects_box.value = subjects_box.value + [new_subject]
    add_subject_input.value = ""  # clear input
    
    
    
def generate_sid_sbd(event):
    print("Inside Generate_SID_SBD Event")
    scenario = scenario_input.value.strip()
    finalized_subjects = "\n".join(f"- {s}" for s in subjects_box.value)
    user_prompt = sid_sbd_user_prompt.replace("{scenario}", scenario).replace("{subjects}", finalized_subjects)
    
    try:
        response = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": sid_sbd_sytem_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            model=model,
            temperature=0.2
        )
        sid_sbd = response.choices[0].message.content.strip()
        
        
        # Build clean text for OWL function
        llm_text = (
            f"### Subjects\n{finalized_subjects}\n\n"
            f"### SID\n{sid_sbd}"
        )

        
        output_markdown.object = (
                                  f"### Subjects:\n{finalized_subjects}\n\n"
                                  f"### SID & SBD Output:\n\n```\n{sid_sbd}\n```"
                                  )
        
         # --- Graph Drawing ---
        sid_pairs = parse_sid(sid_sbd)
        process = {
                    "SID": sid_pairs
                  }
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
        graph_path = fr"C:\Users\qwerz\Documents\Privat\WWU\Wirtschaftsinformatik\Semester 7\Bachelorarbeit\PASS Diagrams\Diagram\gpt-oss\sid_graph-{timestamp}"
        print("Graph saved at:", graph_path + ".png")
        draw_sid_graph(process["SID"],output_path=graph_path)
        
        import os
        print("Exists:", os.path.exists(graph_path + ".png"))

        
        with open(graph_path + ".png", "rb") as f:
                 image_view.object = f.read()
                
      
        
        sbd_textt = extract_sbd_section(sid_sbd)
        
        if sbd_textt:
            print(sbd_textt)
            output_folder = fr"C:\Users\qwerz\Documents\Privat\WWU\Wirtschaftsinformatik\Semester 7\Bachelorarbeit\PASS Diagrams\Diagram\gpt-oss"
            draw_all_subjects(sbd_textt,output_folder)
            
        else:
            print("No SBD section found in response!")
        
        print("llm_text",llm_text)
        xml = sid_to_pass_owl(llm_text, out_file=r"C:\Users\qwerz\Documents\Privat\WWU\Wirtschaftsinformatik\Semester 7\Bachelorarbeit\PASS Diagrams\Diagram\gpt-oss\OWLFile-GPT_OSS.owl")
        
        
        
    except Exception as e:
        output_markdown.object += f"\n\n Error: {e}"

# --- Button bindings ---
extract_btn.on_click(extract_subjects)
add_subject_btn.on_click(add_subject)
apply_merge_btn.on_click(apply_merge)
generate_btn.on_click(generate_sid_sbd)

# --- Layout ---
app = pn.Column(
    pn.pane.Markdown("# PASS Process Modeling with LLM"),
    scenario_input,
    pn.Row(extract_btn, generate_btn),
    subjects_box,
    pn.Row(add_subject_input, add_subject_btn),
    pn.Row(rename_input, apply_merge_btn),
    output_markdown,
    pn.pane.Markdown("### SID Graph:"),
    image_view,
    sizing_mode="stretch_width"
)

app.show()